# 🛒 Solución Guiada: Registro de Ventas con Archivos en Python
### Taller de Programación II — Universidad San Sebastián (Sede Patagonia)

**Asignatura:** Taller de Programación II  
**Fecha:** 6 de Agosto de 2026  
**Estudiante:** Moisés Amundarain Romero  

---

## 📌 Objetivos de la Actividad Práctica
1. Representar cada registro de venta mediante un **diccionario de datos (`dict`)** con 6 claves obligatorias.
2. Implementar funciones modulares con responsabilidades únicas: solicitar, calcular, guardar, leer, mostrar y resumir.
3. Aplicar validaciones de calidad de datos (**Garbage In, Garbage Out - GIGO**).
4. Garantizar la **persistencia de datos** en un archivo `ventas.txt` delimitado por punto y coma (`;`).
5. Usar las buenas prácticas de Python (`with open`, `encoding='utf-8'`, `try-except` y PEP 8).


## 1. Configuración Inicial e Importación de Módulos
Importamos la librería `os` para verificar la existencia física de archivos y `datetime.date` para obtener la fecha de transacción.

In [ ]:
# Importación de módulos nativos de Python
import os
from datetime import date

# Constante global para la ruta del archivo persistente
NOMBRE_ARCHIVO = "ventas.txt"

print(f"Entorno configurado. Archivo de destino: '{NOMBRE_ARCHIVO}'")

## 2. Construcción de Funciones Modulares

### Función 1: `calcular_total(valor_unitario, cantidad)`
Calcula el producto entre el valor unitario en CLP y las unidades vendidas.

In [ ]:
def calcular_total(valor_unitario: int, cantidad: int) -> int:
    """
    Recibe el valor unitario y la cantidad (ambos enteros) y retorna
    el total calculado (valor_unitario * cantidad).
    """
    # Multiplicación directa de números enteros (pesos chilenos)
    return valor_unitario * cantidad

# Prueba rápida de la función
print("Prueba calcular_total(2500, 2):", calcular_total(2500, 2))

### Función 2: `solicitar_venta()`
Solicita interactivamente los datos al usuario o permite la simulación con validaciones estrictas.

In [ ]:
def solicitar_venta_interactiva() -> dict:
    """
    Solicita datos por consola con validaciones completas GIGO.
    Retorna un diccionario con las 6 claves requeridas.
    """
    print("\n--- REGISTRAR NUEVA VENTA (INTERACTIVO) ---")

    # 1. Cliente (No vacío)
    while True:
        cliente = input("Nombre del cliente: ").strip()
        if cliente:
            break
        print("❌ Error: El cliente no puede estar vacío.")

    # 2. Producto (No vacío)
    while True:
        producto = input("Producto: ").strip()
        if producto:
            break
        print("❌ Error: El producto no puede estar vacío.")

    # 3. Valor unitario (Entero > 0)
    while True:
        try:
            valor_unitario = int(input("Valor unitario ($CLP): "))
            if valor_unitario > 0:
                break
            print("❌ Error: El valor unitario debe ser mayor a 0.")
        except ValueError:
            print("❌ Error: Ingrese un número entero válido.")

    # 4. Cantidad (Entero > 0)
    while True:
        try:
            cantidad = int(input("Cantidad: "))
            if cantidad > 0:
                break
            print("❌ Error: La cantidad debe ser mayor a 0.")
        except ValueError:
            print("❌ Error: Ingrese un número entero válido.")

    # 5. Total calculado automáticamente
    total = calcular_total(valor_unitario, cantidad)

    # 6. Fecha actual
    fecha_actual = date.today().strftime("%Y-%m-%d")

    return {
        "nombre_cliente": cliente,
        "producto": producto,
        "valor_unitario": valor_unitario,
        "cantidad": cantidad,
        "total": total,
        "fecha": fecha_actual
    }

### Función 3: `guardar_venta(venta, ruta_archivo)`
Guarda el diccionario formateado en `ventas.txt` utilizando `with open` en **modo append (`'a'`)**.

In [ ]:
def guardar_venta(venta: dict, ruta_archivo: str = NOMBRE_ARCHIVO) -> None:
    """
    Escribe un diccionario de venta en el archivo ventas.txt separados por ';'
    Usa modo 'a' para no sobrescribir registros anteriores.
    """
    # Construcción de la línea delimitada por ';'
    linea = (
        f"{venta['nombre_cliente']};"
        f"{venta['producto']};"
        f"{venta['valor_unitario']};"
        f"{venta['cantidad']};"
        f"{venta['total']};"
        f"{venta['fecha']}\n"
    )

    # Apertura segura en modo 'a' (append) y codificación UTF-8
    with open(ruta_archivo, "a", encoding="utf-8") as archivo:
        archivo.write(linea)
    
    print(f"💾 Registro de '{venta['producto']}' guardado en '{ruta_archivo}'.")

### Función 4: `leer_ventas(ruta_archivo)`
Lee el archivo `ventas.txt`, limpia saltos de línea con `.strip()`, desestructura los campos con `.split(';')`, convierte tipos y reconstruye diccionarios.

In [ ]:
def leer_ventas(ruta_archivo: str = NOMBRE_ARCHIVO) -> list:
    """
    Lee el archivo ventas.txt y retorna una lista de diccionarios con las ventas.
    """
    ventas = []

    if not os.path.exists(ruta_archivo):
        print(f"⚠️ El archivo '{ruta_archivo}' aún no existe.")
        return ventas

    with open(ruta_archivo, "r", encoding="utf-8") as archivo:
        for linea in archivo:
            linea_limpia = linea.strip()
            if not linea_limpia:
                continue
            partes = linea_limpia.split(";")
            if len(partes) == 6:
                registro = {
                    "nombre_cliente": partes[0],
                    "producto": partes[1],
                    "valor_unitario": int(partes[2]),
                    "cantidad": int(partes[3]),
                    "total": int(partes[4]),
                    "fecha": partes[5]
                }
                ventas.append(registro)
    return ventas

### Función 5 y 6: `mostrar_ventas(ventas)` y `calcular_resumen(ventas)`
Muestra los registros en tabla y calcula el acumulado total.

In [ ]:
def mostrar_ventas(ventas: list) -> None:
    """Muestra las ventas recuperadas en un formato tabulado."""
    print("\n" + "=" * 85)
    print(f"{'N°':<3} | {'CLIENTE':<20} | {'PRODUCTO':<18} | {'UNITARIO':<9} | {'CANT':<4} | {'TOTAL':<9} | {'FECHA':<10}")
    print("=" * 85)
    for idx, v in enumerate(ventas, start=1):
        print(
            f"{idx:<3} | "
            f"{v['nombre_cliente']:<20} | "
            f"{v['producto']:<18} | "
            f"${v['valor_unitario']:<8,d} | "
            f"{v['cantidad']:<4} | "
            f"${v['total']:<8,d} | "
            f"{v['fecha']:<10}"
        )
    print("=" * 85)

def calcular_resumen(ventas: list) -> dict:
    """Aplica el patrón acumulador para calcular cantidad y total general."""
    cantidad_ventas = len(ventas)
    total_general = sum(v["total"] for v in ventas)

    print("\n=== RESUMEN DE JORNADA DE VENTAS ===")
    print(f" Total transacciones : {cantidad_ventas}")
    print(f" Monto Acumulado     : ${total_general:,.0f} CLP")
    print("====================================")
    return {"cantidad_ventas": cantidad_ventas, "total_general": total_general}

## 3. Demostración Automatizada y Prueba de Persistencia
A continuación, crearemos **3 ventas de prueba requeridas** por la pauta de evaluación, las guardaremos en `ventas.txt`, luego las leeremos desde el disco y generaremos el resumen.

In [ ]:
# Limpieza inicial si existe el archivo previo para prueba controlada
if os.path.exists(NOMBRE_ARCHIVO):
    os.remove(NOMBRE_ARCHIVO)
    print(f"Archivo previo '{NOMBRE_ARCHIVO}' removido para prueba limpia.")

# 1. Definición de 3 ventas de prueba según requerimiento de la pauta
ventas_demo = [
    {
        "nombre_cliente": "María González",
        "producto": "Pan amasado",
        "valor_unitario": 2500,
        "cantidad": 2,
        "total": calcular_total(2500, 2),
        "fecha": "2026-08-06"
    },
    {
        "nombre_cliente": "Juan Pérez",
        "producto": "Café",
        "valor_unitario": 1800,
        "cantidad": 1,
        "total": calcular_total(1800, 1),
        "fecha": "2026-08-06"
    },
    {
        "nombre_cliente": "Carlos Tapia",
        "producto": "Jugo Natural",
        "valor_unitario": 2200,
        "cantidad": 3,
        "total": calcular_total(2200, 3),
        "fecha": "2026-08-06"
    }
]

# 2. Guardar cada venta en ventas.txt
print("\n--- GUARDANDO 3 VENTAS EN EL ARCHIVO ---")
for v in ventas_demo:
    guardar_venta(v)

# 3. Leer las ventas desde el archivo persistente ventas.txt
print("\n--- LEYENDO Y RECONSTRUYENDO DESDE VENTAS.TXT ---")
ventas_recuperadas = leer_ventas()

# 4. Mostrar la tabla recuperada
mostrar_ventas(ventas_recuperadas)

# 5. Calcular y mostrar resumen
calcular_resumen(ventas_recuperadas)

## 4. Inspección Directa del Archivo Creado (`ventas.txt`)
Comprobamos la estructura exacta de las líneas físicas guardadas en el disco.

In [ ]:
print(f"📄 Contenido físico de '{NOMBRE_ARCHIVO}':\n")
with open(NOMBRE_ARCHIVO, "r", encoding="utf-8") as f:
    print(f.read())